# analisis dataset pada world_video_games_final.parquet

In [139]:
import pandas as pd
import numpy as np
import pyarrow
import fastparquet
import ast

df = pd.read_parquet('world_video_games_final.parquet')

In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4222 entries, 0 to 4221
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   title         4222 non-null   object        
 1   release_date  4138 non-null   datetime64[ns]
 2   platforms     4222 non-null   object        
 3   genres        4222 non-null   object        
dtypes: datetime64[ns](1), object(3)
memory usage: 132.1+ KB


## bagaimana perkembangan jumlah game dari tahun ke tahun?

### analisis jumlah game pertahun

membuat kolom release_year untuk spesifik menganlisis tahun

In [140]:
df['release_year'] = df['release_date'].dt.year.astype('Int64')

In [114]:
df[['title', 'release_date', 'release_year']].head()

,title,release_date,release_year
0,.kkrieger,2004-01-01,2004
1,0 A.D.,2021-02-21,2021
2,007 Legends,2012-10-16,2012
3,007: Licence to Kill,1989-04-20,1989
4,007: Quantum of Solace,2008-10-31,2008


menghitung jumlah game setiap tahun

In [141]:
game_per_year = (
    df.groupby('release_year')
    .size()
    .reset_index(name = "total_games")
)

game_per_year.head()

,release_year,total_games
0,1951,1
1,1952,1
2,1958,1
3,1962,1
4,1969,2


In [142]:
game_per_year

,release_year,total_games
0,1951,1
1,1952,1
2,1958,1
3,1962,1
4,1969,2
5,1971,3
6,1972,1
7,1973,2
8,1974,4
9,1975,4


### tahun dengan jumlah game terbanyak

menghitung tahun dengan jumlah game terbanyak

In [143]:
max_year = game_per_year.loc[
    game_per_year['total_games'].idxmax()
]
print("tahun dengan jumlah game terbanyak : ")
print(max_year)

tahun dengan jumlah game terbanyak : 
release_year    2007
total_games      233
Name: 41, dtype: Int64


menghitung tahun dengan jumlah game paling sedikit

In [144]:
min_year = game_per_year.loc[
    game_per_year['total_games'].idxmin()
]
print("tahun dengan jumlah game paling sedikit : ")
print(min_year)

tahun dengan jumlah game paling sedikit : 
release_year    1951
total_games        1
Name: 0, dtype: Int64


### melihat 10 tahun dengan jumlah game terbanyak

In [145]:
top_10_years = (
    game_per_year
    .sort_values("total_games", ascending = False)
    .head(10)
)
top_10_years

,release_year,total_games
41,2007,233
40,2006,223
42,2008,217
39,2005,205
38,2004,201
43,2009,187
37,2003,185
44,2010,181
35,2001,151
33,1999,142


## genre apa yang paling banyak digunakan?

### analisis jumlah game berdasarkan genre

In [146]:
game_per_genre = df['genres'].value_counts()
game_per_genre

genres
[platformer]                                                                                              204
[action-adventure game]                                                                                   178
[role-playing video game]                                                                                 175
[racing video game]                                                                                       171
[action game]                                                                                             150
                                                                                                         ... 
[harem, magical girl]                                                                                       1
[game creation system, puzzle video game]                                                                   1
[adventure video game, science fiction video game]                                                          1
[ac

### top 10 genre

In [102]:
game_per_genre.head(10)

genres
[platformer]                 204
[action-adventure game]      178
[role-playing video game]    175
[racing video game]          171
[action game]                150
[shoot 'em up]               146
[first-person shooter]       144
[adventure video game]       123
[fighting game]              107
[puzzle video game]           98
Name: count, dtype: int64

### persentase setiap genre

In [147]:
persentase_genre = df['genres'].value_counts(normalize=True) * 100
print(persentase_genre.map("{:.2f}%".format))

genres
[platformer]                                                                                              4.83%
[action-adventure game]                                                                                   4.22%
[role-playing video game]                                                                                 4.14%
[racing video game]                                                                                       4.05%
[action game]                                                                                             3.55%
                                                                                                          ...  
[harem, magical girl]                                                                                     0.02%
[game creation system, puzzle video game]                                                                 0.02%
[adventure video game, science fiction video game]                                               

## platform apa yang paling banyak digunakan?

### jumlah game berdasarkan platform 

In [150]:
game_per_platform = df['platforms'].value_counts()
game_per_platform

platforms
[Microsoft Windows]                                                                                                                   592
[PlayStation 4]                                                                                                                       138
[Unknown]                                                                                                                             121
[Nintendo DS]                                                                                                                         107
[Nintendo Entertainment System]                                                                                                        77
                                                                                                                                     ... 
[PC Engine SuperGrafx, TurboGrafx-16, arcade video game]                                                                                1
[Amstrad CPC, Atari ST, 

### top 10 platform

In [149]:
game_per_platform.head(10)

platforms
[Microsoft Windows]                592
[PlayStation 4]                    138
[Unknown]                          121
[Nintendo DS]                      107
[Nintendo Entertainment System]     77
[PlayStation 2]                     77
[arcade video game]                 73
[Wii]                               65
[DOS]                               54
[PlayStation Portable]              43
Name: count, dtype: int64

### perkembangan platform dari waktu ke waktu

In [156]:
df_exploded = df.explode('platforms')

df_exploded['platforms'] = df_exploded['platforms'].astype(str).str.strip()

game_per_platform_by_year = df_exploded.groupby(['release_year', 'platforms']).size().reset_index(name="jumlah_game")

game_per_platform_by_year


,release_year,platforms,jumlah_game
0,1951,http://www.wikidata.org/.well-known/genid/b1cd...,1
1,1952,Electronic Delay Storage Automatic Calculator,1
2,1958,Donner Model 30,1
3,1962,PDP-1,1
4,1969,General Comprehensive Operating System,1
...,...,...,...
1414,2023,Microsoft Windows,2
1415,2023,Xbox Series X and Series S,1
1416,2023,ZX Spectrum,1
1417,2024,Microsoft Windows,2
